In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

wanted_issues = [
    "Amalgamated with 12559, then Delete"]

df_issue = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_issue = df_issue[['ISSUE','Network ID', 'Network Name', 'Station ID',  'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

df_issue["old_station_id"] = pd.to_numeric(df_issue["Station ID"], errors="coerce").astype("Int64")


df_issue["new_station_id"] = 12559
# df_issue["new_network_id"] = 141

In [5]:
# native_id AFU corresponds to the station 12342 under 5,BCH-GSO-HMP.

In [6]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND m_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_station_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_station_id,

        # n_new
        new_station_id,

        # n_overlap
        old_station_id,
        new_station_id,

        # n_overlap_same_datum
        old_station_id,
        new_station_id,
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = df_issue.apply(
    lambda r: count_station_stats(
        r["old_station_id"],
        r["new_station_id"],
        engine
    ),
    axis=1
)

df_issue[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

In [7]:
df_issue

,ISSUE,Network ID,Network Name,Station ID,Native ID,Unique Names,Unique Location (String),old_station_id,new_station_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,"Amalgamated with 12559, then Delete",9.0,BC-Air -> BCH-SiteC,12213.0,E304453 -> AFU,Peace Valley Attachie Flat Upper Terrace_60 ->...,"121.41944 W, 56.231213 N, Elev. 480 m",12213,12559,206975,2819,572,572


In [8]:
from sqlalchemy import text

SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :new_station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_MOVE = text("""
WITH updated AS (
    UPDATE obs_raw o
    SET history_id = :new_hist_id
    FROM meta_history h_old
    WHERE o.history_id = h_old.history_id
      AND h_old.station_id = :old_station_id

      AND NOT EXISTS (
          SELECT 1
          FROM obs_raw o_new
          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
          WHERE h_new.station_id = :new_station_id
            AND o_new.obs_time = o.obs_time
            AND o_new.vars_id  = o.vars_id
      )
    RETURNING 1
)
SELECT COUNT(*) FROM updated;
""")

def move_station_obs_fast(
    old_station_id,
    new_station_id,
    engine
):
    with engine.begin() as conn:
        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {"new_station_id": new_station_id}
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_station_id} has no history, skipping.")
            return 0

        n_move = conn.execute(
            SQL_MOVE,
            {
                "old_station_id": old_station_id,
                "new_station_id": new_station_id,
                "new_hist_id": new_hist_id,
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from station {old_station_id} → {new_station_id}"
        )

        return n_move

results = []

for _, row in df_issue.iterrows():
    n = move_station_obs_fast(
        row["old_station_id"],
        row["new_station_id"],
        engine
    )
    results.append(n)

df_issue["n_moved"] = results

Moved 206403 rows from station 12213 → 12559


### Delete old station

In [9]:
old_station_id = 12213
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 12213

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 12213: flags=0, obs_raw=572, obs_derived=0, meta_history=1, meta_station=1


### Rename the new station's native id

In [10]:
df_issue['old_station_id']

0    12213
Name: old_station_id, dtype: Int64

In [11]:
select_sql = sa.text("""
SELECT DISTINCT
    n.network_id
FROM meta_station s
JOIN meta_network n
  ON n.network_id = s.network_id
JOIN meta_history m
  ON m.station_id = s.station_id
WHERE s.station_id = :station_id
""")

station_id = str(df_issue.loc[0, 'new_station_id'])

with engine.begin() as conn:
    result = conn.execute(select_sql, {"station_id": station_id})
    row = result.fetchone()  # returns a tuple-like row
    if row:
        old_net_id = row[0]   # first column
        print(old_net_id)
    else:
        print("No network_id found for this station_id")

70


### Check the data condition of station '12559'

### Change the '12559' station to new network 'BCH-SiteC', and change the native_id to AFU, name to Peace Valley Attachie Flat Upper Terrace

In [12]:
station_id

from sqlalchemy import text

# 1️⃣ Update meta_station
SQL_UPDATE_STATION = text("""
UPDATE meta_station s
SET
    native_id = :new_native_id,
    network_id = (
        SELECT n.network_id
        FROM meta_network n
        WHERE n.network_name = :new_network_name
        LIMIT 1
    )
WHERE s.station_id = :station_id
""")

# 2️⃣ Update meta_history
SQL_UPDATE_HISTORY = text("""
UPDATE meta_history h
SET station_name = :new_station_name
WHERE h.station_id = :station_id
""")

station_id = 12559

with engine.begin() as conn:
    # Update meta_station
    res1 = conn.execute(
        SQL_UPDATE_STATION,
        {
            "station_id": station_id,
            "new_native_id": "AFU",
            "new_network_name": "BCH-SiteC",
        }
    )
    print(f"meta_station updated, rows affected: {res1.rowcount}")

    # Update meta_history
    res2 = conn.execute(
        SQL_UPDATE_HISTORY,
        {
            "station_id": station_id,
            "new_station_name": "Peace Valley Attachie Flat Upper Terrace",
        }
    )
    print(f"meta_history updated, rows affected: {res2.rowcount}")

meta_station updated, rows affected: 1
meta_history updated, rows affected: 1


In [13]:
from sqlalchemy import text
import pandas as pd

station_id = 12559

# 1️⃣ Check meta_station
SQL_CHECK_STATION = text("""
SELECT s.station_id, s.native_id, n.network_name
FROM meta_station s
JOIN meta_network n ON s.network_id = n.network_id
WHERE s.station_id = :station_id
""")

df_station = pd.read_sql(SQL_CHECK_STATION, engine, params={"station_id": station_id})
print("meta_station info:")
print(df_station)

# 2️⃣ Check meta_history
SQL_CHECK_HISTORY = text("""
SELECT h.history_id, h.station_name
FROM meta_history h
WHERE h.station_id = :station_id
""")

df_history = pd.read_sql(SQL_CHECK_HISTORY, engine, params={"station_id": station_id})
print("\nmeta_history info:")
print(df_history)

meta_station info:
   station_id native_id network_name
0       12559       AFU    BCH-SiteC

meta_history info:
   history_id                              station_name
0       14581  Peace Valley Attachie Flat Upper Terrace


In [14]:
from sqlalchemy import text
import pandas as pd

SQL_VARS_INFO = text("""
SELECT DISTINCT
       -- o.vars_id,
       v.*,
       -- m.station_id,
       s.native_id
       -- m.station_name,
       -- s.network_id AS station_network_id
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
JOIN obs_raw AS o ON m.history_id = o.history_id
JOIN meta_vars AS v ON o.vars_id = v.vars_id
WHERE s.station_id = :station_id
""")

old_station_vars = pd.read_sql(
    SQL_VARS_INFO,
    engine,
    params={"station_id": 12559}
)

old_station_vars

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id
0,474,9,%,None,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point,2025-02-11 16:03:39.747374,crmp,AFU
1,477,9,m/s,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point,2025-02-11 16:03:39.747374,crmp,AFU
2,699,9,%,None,relative_humidity,time: mean,None,avg_rel_hum_pst1hr,Relative Humidity,rh,2025-02-11 16:03:39.747374,crmp,AFU
3,475,9,millibar,None,air_pressure,time: point,Atmospheric pressure,BAR_PRESS,Air Pressure (Point),air_pressure_point,2025-02-11 16:03:39.747374,crmp,AFU
4,472,9,celsius,None,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean,2025-02-11 16:03:39.747374,crmp,AFU
5,693,9,celsius,None,air_temperature,time: mean,None,avg_air_temp_pst1hr,Temperature (Mean),T,2025-02-11 16:03:39.747374,crmp,AFU


In [15]:
from sqlalchemy import text
import pandas as pd

SQL_VARS_new = text("""
SELECT DISTINCT
       v.vars_id,
       v.unit,
       v.precision,
       v.standard_name,
       v.cell_method,
       v.long_description,
       v.net_var_name,
       v.display_name,
       v.short_name
FROM meta_vars v
JOIN meta_network n ON v.network_id = n.network_id
WHERE n.network_name = :network_name
""")

new_net_vars = pd.read_sql(
    SQL_VARS_new,
    engine,
    params={"network_name": "BCH-SiteC"}
)

new_net_vars.head()

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name


In [16]:
VAR_MATCH_COLS = [
    "unit",
    "precision",
    "standard_name",
    "cell_method",
    "long_description",
    "net_var_name",
    "display_name",
    "short_name",
]

# Merge again, but keep vars_id from network 14
df_compare = old_station_vars.merge(
    new_net_vars[VAR_MATCH_COLS + ["vars_id"]],  # only keep vars_id from new_net
    on=VAR_MATCH_COLS,
    how="left",
    suffixes=("", "_new_net"),
    indicator=True
)

# Add a boolean column to mark existence
df_compare["exists_in_new_net"] = df_compare["_merge"] == "both"

# Rename vars_id from network 14 for clarity
df_compare["vars_id_new_net"] = df_compare["vars_id_new_net"]

df_compare

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id,vars_id_new_net,_merge,exists_in_new_net
0,474,9,%,None,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point,2025-02-11 16:03:39.747374,crmp,AFU,NaN,left_only,False
1,477,9,m/s,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point,2025-02-11 16:03:39.747374,crmp,AFU,NaN,left_only,False
2,699,9,%,None,relative_humidity,time: mean,None,avg_rel_hum_pst1hr,Relative Humidity,rh,2025-02-11 16:03:39.747374,crmp,AFU,NaN,left_only,False
3,475,9,millibar,None,air_pressure,time: point,Atmospheric pressure,BAR_PRESS,Air Pressure (Point),air_pressure_point,2025-02-11 16:03:39.747374,crmp,AFU,NaN,left_only,False
4,472,9,celsius,None,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean,2025-02-11 16:03:39.747374,crmp,AFU,NaN,left_only,False
5,693,9,celsius,None,air_temperature,time: mean,None,avg_air_temp_pst1hr,Temperature (Mean),T,2025-02-11 16:03:39.747374,crmp,AFU,NaN,left_only,False


One station might have part of variabes of that network.

so when reassigning the station to new network, we deal with it station by station. First to move the station's records. Then check if the variable is in the new station, if not, copy (create) the variable, generate new variable id; if yes, update the variable ID



In [17]:
from sqlalchemy import text

new_network_name = "BCH-SiteC"

SQL_GET_NETWORK_ID = text("""
SELECT n.network_id
FROM meta_network n
WHERE n.network_name = :new_network_name
LIMIT 1
""")

with engine.begin() as conn:
    new_network_id = conn.execute(
        SQL_GET_NETWORK_ID,
        {"new_network_name": new_network_name}
    ).scalar()  # .scalar() returns the first column of the first row

print(f"Network ID for '{new_network_name}' is {new_network_id}")

Network ID for 'BCH-SiteC' is 70


In [12]:
new_network_id



70

In [18]:
from pycds import Variable
help(Variable)


Help on class Variable in module pycds.orm.tables:

class Variable(sqlalchemy.orm.decl_api.Base)
 |  Variable(**kwargs)
 |
 |  This class maps to the table which records the details of the
 |  physical quantities which are recorded by the weather stations.
 |
 |  Method resolution order:
 |      Variable
 |      sqlalchemy.orm.decl_api.Base
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, **kwargs) from sqlalchemy.orm.instrumentation
 |      A simple constructor that allows initialization from kwargs.
 |
 |      Sets attributes on the constructed instance using the names and
 |      values in ``kwargs``.
 |
 |      Only keys that are present as
 |      attributes of the instance's class are allowed. These could be,
 |      for example, any mapped columns or relationships.
 |
 |  __repr__(self)
 |      Return repr(self).
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  cell_method
 |
 |  der

In [18]:
from pycds import Variable

vars_created = []
vars_map = {}   # old_vars_id → new_vars_id

for _, row in old_station_vars.iterrows():

    v = Variable(
        network_id=new_network_id,
        unit=row["unit"],
        precision=row["precision"],
        standard_name=row["standard_name"],
        cell_method=row["cell_method"],
        description=row["long_description"],
        name=row["net_var_name"],
        display_name=row["display_name"],
        short_name=row["short_name"],
    )

    session.add(v)
    session.flush()   # assigns vars_id

    vars_created.append(v.id)
    vars_map[row["vars_id"]] = v.id   # ← critical for later obs reassignment

session.commit()

print(f"Created {len(vars_created)} variables for network {new_network_id}")

Created 6 variables for network 70


In [19]:
vars_map

{474: 1301, 477: 1302, 699: 1303, 475: 1304, 472: 1305, 693: 1306}

### Update the vars id to the new one

In [20]:
import json
from sqlalchemy import text

pairs_json = json.dumps(
    [{"old_vars_id": old, "new_vars_id": new}
     for old, new in vars_map.items()]
)

SQL_UPDATE_VARS_BULK = text("""
UPDATE obs_raw o
SET vars_id = m.new_vars_id
FROM meta_history h
JOIN meta_station s
  ON h.station_id = s.station_id
CROSS JOIN json_to_recordset(:pairs_json)
     AS m(old_vars_id INT, new_vars_id INT)
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND o.vars_id = m.old_vars_id
""")

new_station_id = 12559

with engine.begin() as conn:
    res = conn.execute(
        SQL_UPDATE_VARS_BULK,
        {
            "pairs_json": pairs_json,
            "station_id": new_station_id,
        }
    )

print(
    f"✅ vars_id remap completed | "
    f"station={new_station_id} | "
    f"mappings={len(vars_map)} | "
    f"rows_updated={res.rowcount}"
)

✅ vars_id remap completed | station=12559 | mappings=6 | rows_updated=209222


In [21]:
from sqlalchemy import text
import pandas as pd

SQL_VARS_new = text("""
SELECT DISTINCT
       v.vars_id,
       v.unit,
       v.precision,
       v.standard_name,
       v.cell_method,
       v.long_description,
       v.net_var_name,
       v.display_name,
       v.short_name,
       v.network_id
FROM meta_vars v
JOIN meta_network n ON v.network_id = n.network_id
WHERE n.network_name = :network_name
""")

new_net_vars = pd.read_sql(
    SQL_VARS_new,
    engine,
    params={"network_name": "BCH-SiteC"}
)

new_net_vars.head()

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,network_id
0,1301,%,None,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point,70
1,1302,m/s,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point,70
2,1303,%,None,relative_humidity,time: mean,None,avg_rel_hum_pst1hr,Relative Humidity,rh,70
3,1304,millibar,None,air_pressure,time: point,Atmospheric pressure,BAR_PRESS,Air Pressure (Point),air_pressure_point,70
4,1305,celsius,None,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean,70
